In [5]:
import os

intermediate_path = "../data/intermediate"

print("Isi folder intermediate:")
for f in os.listdir(intermediate_path):
    print(f)

Isi folder intermediate:
fire_kalimantan_cell_day_full.csv
fire_kalimantan_cell_day_poc_2026-09-07.csv
fire_kalimantan_cell_day_poc_5days.csv
weather_kalimantan_rich_daily.csv


In [6]:
import pandas as pd

weather_rich_daily = pd.read_csv(
    "../data/intermediate/weather_kalimantan_rich_daily.csv"
)

fire_panel = pd.read_csv(
    "../data/intermediate/fire_kalimantan_cell_day_full.csv"
)

print("Weather:", weather_rich_daily.shape)
print("Fire:", fire_panel.shape)

Weather: (13400, 14)
Fire: (6524, 6)


In [7]:
print(fire_panel.columns.tolist())
print(fire_panel.head())

['cell_id', 'date', 'n_detections', 'frp_sum', 'frp_mean', 'frp_max']
        cell_id        date  n_detections  frp_sum  frp_mean  frp_max
0   -0.75_117.0  2026-05-01             1     1.01  1.010000     1.01
1  -0.75_117.75  2026-05-01             1     0.33  0.330000     0.33
2    -1.5_117.0  2026-05-01             3     6.31  2.103333     2.65
3   -2.25_115.5  2026-05-01             5     5.90  1.180000     1.67
4    -3.0_115.5  2026-05-01             2     3.35  1.675000     1.96


In [8]:
print(
    "Fire:",
    fire_panel["date"].min(),
    "→",
    fire_panel["date"].max()
)

print(
    "Weather:",
    weather_rich_daily["date"].min(),
    "→",
    weather_rich_daily["date"].max()
)

Fire: 2026-05-01 → 2026-09-11
Weather: 2026-05-01 → 2026-09-11


In [9]:
aq_vars = [
    "pm2_5",
    "aerosol_optical_depth",
    "carbon_monoxide",
    "formaldehyde"
]

print("Variabel AQ:")
for v in aq_vars:
    print("-", v)

Variabel AQ:
- pm2_5
- aerosol_optical_depth
- carbon_monoxide
- formaldehyde


In [10]:
def process_aq_rich_file(file_path):
    df = pd.read_csv(file_path)

    # Ambil variabel yang dipakai
    df = df[df["variable"].isin(aq_vars)].copy()

    df["time_utc"] = pd.to_datetime(
        df["time_utc"],
        errors="coerce"
    )

    df["date"] = df["time_utc"].dt.date

    # Long -> wide
    df_wide = (
        df.pivot_table(
            index=["time_utc", "lat", "lon", "date"],
            columns="variable",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )

    # Pastikan semua variable tersedia
    for col in aq_vars:
        if col not in df_wide.columns:
            df_wide[col] = np.nan

    # Daily aggregation
    daily = (
        df_wide
        .groupby(["lat", "lon", "date"])
        .agg(
            pm2_5_mean=("pm2_5", "mean"),
            pm2_5_max=("pm2_5", "max"),

            aerosol_optical_depth_mean=(
                "aerosol_optical_depth",
                "mean"
            ),

            carbon_monoxide_mean=(
                "carbon_monoxide",
                "mean"
            ),

            formaldehyde_mean=(
                "formaldehyde",
                "mean"
            )
        )
        .reset_index()
    )

    return daily

In [12]:
import glob
import os
import numpy as np
import pandas as pd

In [13]:
aq_rich_files = sorted(
    glob.glob(
        "../data/raw/kalimantan/air_quality_openmeteo/hourly_rich/*.csv"
    )
)

print("Jumlah file:", len(aq_rich_files))

test_aq = process_aq_rich_file(aq_rich_files[0])

print("\nShape:", test_aq.shape)

print("\nKolom:")
print(test_aq.columns.tolist())

print("\nMissing:")
print(test_aq.isna().sum())

Jumlah file: 134

Shape: (272, 8)

Kolom:
['lat', 'lon', 'date', 'pm2_5_mean', 'pm2_5_max', 'aerosol_optical_depth_mean', 'carbon_monoxide_mean', 'formaldehyde_mean']

Missing:
lat                           0
lon                           0
date                          0
pm2_5_mean                    0
pm2_5_max                     0
aerosol_optical_depth_mean    0
carbon_monoxide_mean          0
formaldehyde_mean             0
dtype: int64


In [14]:
aq_daily_list = []

for i, file_path in enumerate(aq_rich_files, start=1):

    daily = process_aq_rich_file(file_path)
    aq_daily_list.append(daily)

    if i % 10 == 0 or i == len(aq_rich_files):
        print(f"Processed {i}/{len(aq_rich_files)} files")

Processed 10/134 files
Processed 20/134 files
Processed 30/134 files
Processed 40/134 files
Processed 50/134 files
Processed 60/134 files
Processed 70/134 files
Processed 80/134 files
Processed 90/134 files
Processed 100/134 files
Processed 110/134 files
Processed 120/134 files
Processed 130/134 files
Processed 134/134 files


In [15]:
aq_rich_daily = pd.concat(
    aq_daily_list,
    ignore_index=True
)

print("Shape:", aq_rich_daily.shape)

print(
    "Tanggal:",
    aq_rich_daily["date"].min(),
    "→",
    aq_rich_daily["date"].max()
)

print(
    "Jumlah grid:",
    aq_rich_daily[["lat", "lon"]]
    .drop_duplicates()
    .shape[0]
)

Shape: (36448, 8)
Tanggal: 2026-05-01 → 2026-09-11
Jumlah grid: 272


In [16]:
print(
    "Duplicate grid-day:",
    aq_rich_daily.duplicated(
        subset=["lat", "lon", "date"]
    ).sum()
)

Duplicate grid-day: 0


In [17]:
print("\nMissing:")
print(aq_rich_daily.isna().sum())


Missing:
lat                           0
lon                           0
date                          0
pm2_5_mean                    0
pm2_5_max                     0
aerosol_optical_depth_mean    0
carbon_monoxide_mean          0
formaldehyde_mean             0
dtype: int64


In [18]:
aq_rich_daily.to_csv(
    "../data/intermediate/aq_kalimantan_rich_daily.csv",
    index=False
)

print("AQ daily saved!")

AQ daily saved!


In [19]:
print(fire_panel.shape)
print(fire_panel.columns.tolist())

print("\nTanggal:")
print(fire_panel["date"].min(), "→", fire_panel["date"].max())

print("\nJumlah cell:")
print(fire_panel["cell_id"].nunique())

(6524, 6)
['cell_id', 'date', 'n_detections', 'frp_sum', 'frp_mean', 'frp_max']

Tanggal:
2026-05-01 → 2026-09-11

Jumlah cell:
166


In [20]:
# Pastikan date bertipe string konsisten
fire_panel["date"] = pd.to_datetime(
    fire_panel["date"]
).dt.strftime("%Y-%m-%d")

weather_rich_daily["date"] = pd.to_datetime(
    weather_rich_daily["date"]
).dt.strftime("%Y-%m-%d")

aq_rich_daily["date"] = pd.to_datetime(
    aq_rich_daily["date"]
).dt.strftime("%Y-%m-%d")

In [21]:
# AQ dan Fire menggunakan grid yang sama
fire_grid = aq_rich_daily[["lat", "lon"]].drop_duplicates().copy()

fire_grid["cell_id"] = (
    fire_grid["lat"].astype(str)
    + "_"
    + fire_grid["lon"].astype(str)
)

print("Jumlah Fire grid:", len(fire_grid))

Jumlah Fire grid: 272


In [22]:
dates = pd.DataFrame({
    "date": sorted(aq_rich_daily["date"].unique())
})

full_fire_panel = (
    fire_grid[["cell_id"]]
    .merge(dates, how="cross")
)

print("Shape:", full_fire_panel.shape)

Shape: (36448, 2)


In [23]:
full_fire_panel = full_fire_panel.merge(
    fire_panel,
    on=["cell_id", "date"],
    how="left"
)

In [24]:
fire_cols = [
    "n_detections",
    "frp_sum",
    "frp_mean",
    "frp_max"
]

full_fire_panel[fire_cols] = (
    full_fire_panel[fire_cols].fillna(0)
)

full_fire_panel["fire_active"] = (
    full_fire_panel["n_detections"] > 0
).astype(int)

In [25]:
print("Shape:", full_fire_panel.shape)

print("\nFire active:")
print(full_fire_panel["fire_active"].value_counts())

print("\nJumlah cell:", full_fire_panel["cell_id"].nunique())

print(
    "\nTanggal:",
    full_fire_panel["date"].min(),
    "→",
    full_fire_panel["date"].max()
)

Shape: (36448, 7)

Fire active:
fire_active
0    29924
1     6524
Name: count, dtype: int64

Jumlah cell: 272

Tanggal: 2026-05-01 → 2026-09-11


In [26]:
full_fire_panel["target_fire_active_t1"] = (
    full_fire_panel
    .groupby("cell_id")["fire_active"]
    .shift(-1)
)

In [27]:
print(
    full_fire_panel["target_fire_active_t1"]
    .value_counts(dropna=False)
)

target_fire_active_t1
0.0    29679
1.0     6497
NaN      272
Name: count, dtype: int64


In [28]:
full_fire_panel = full_fire_panel.sort_values(
    ["cell_id", "date"]
).reset_index(drop=True)

g = full_fire_panel.groupby("cell_id")

# Lag
full_fire_panel["fire_count_lag1"] = (
    g["n_detections"].shift(1)
)

full_fire_panel["fire_count_lag3"] = (
    g["n_detections"].shift(3)
)

full_fire_panel["fire_count_lag7"] = (
    g["n_detections"].shift(7)
)

full_fire_panel["frp_sum_lag1"] = (
    g["frp_sum"].shift(1)
)

full_fire_panel["frp_sum_lag3"] = (
    g["frp_sum"].shift(3)
)

full_fire_panel["frp_sum_lag7"] = (
    g["frp_sum"].shift(7)
)

In [29]:
# Rolling hanya menggunakan hari-hari SEBELUM hari prediksi
full_fire_panel["fire_count_roll3"] = (
    g["n_detections"]
    .shift(1)
    .groupby(full_fire_panel["cell_id"])
    .rolling(3)
    .sum()
    .reset_index(level=0, drop=True)
)

full_fire_panel["fire_count_roll7"] = (
    g["n_detections"]
    .shift(1)
    .groupby(full_fire_panel["cell_id"])
    .rolling(7)
    .sum()
    .reset_index(level=0, drop=True)
)

full_fire_panel["frp_sum_roll3"] = (
    g["frp_sum"]
    .shift(1)
    .groupby(full_fire_panel["cell_id"])
    .rolling(3)
    .sum()
    .reset_index(level=0, drop=True)
)

full_fire_panel["frp_sum_roll7"] = (
    g["frp_sum"]
    .shift(1)
    .groupby(full_fire_panel["cell_id"])
    .rolling(7)
    .sum()
    .reset_index(level=0, drop=True)
)

In [30]:
history_cols = [
    "fire_count_lag1",
    "fire_count_lag3",
    "fire_count_lag7",
    "frp_sum_lag1",
    "frp_sum_lag3",
    "frp_sum_lag7",
    "fire_count_roll3",
    "fire_count_roll7",
    "frp_sum_roll3",
    "frp_sum_roll7"
]

print(full_fire_panel[history_cols].isna().sum())

fire_count_lag1      272
fire_count_lag3      816
fire_count_lag7     1904
frp_sum_lag1         272
frp_sum_lag3         816
frp_sum_lag7        1904
fire_count_roll3     816
fire_count_roll7    1904
frp_sum_roll3        816
frp_sum_roll7       1904
dtype: int64


In [31]:
# Mapping Fire cell ke koordinat AQ/Fire grid
fire_cell_coords = (
    aq_rich_daily[["lat", "lon"]]
    .drop_duplicates()
    .copy()
)

fire_cell_coords["cell_id"] = (
    fire_cell_coords["lat"].astype(str)
    + "_"
    + fire_cell_coords["lon"].astype(str)
)

print("Fire cells:", len(fire_cell_coords))

Fire cells: 272


In [32]:
weather_grid = (
    weather_rich_daily[["lat", "lon"]]
    .drop_duplicates()
    .copy()
)

print("Weather grids:", len(weather_grid))

Weather grids: 100


In [33]:
from math import radians, sin, cos, asin, sqrt

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    return 2 * R * asin(sqrt(a))

In [34]:
mapping_rows = []

for _, fire_row in fire_cell_coords.iterrows():

    distances = weather_grid.apply(
        lambda w: haversine_km(
            fire_row["lat"],
            fire_row["lon"],
            w["lat"],
            w["lon"]
        ),
        axis=1
    )

    nearest_idx = distances.idxmin()
    nearest_weather = weather_grid.loc[nearest_idx]

    mapping_rows.append({
        "cell_id": fire_row["cell_id"],
        "fire_lat": fire_row["lat"],
        "fire_lon": fire_row["lon"],
        "weather_lat": nearest_weather["lat"],
        "weather_lon": nearest_weather["lon"],
        "distance_km": distances.loc[nearest_idx]
    })

weather_mapping = pd.DataFrame(mapping_rows)

print(weather_mapping.shape)
print(weather_mapping.head())
print("\nDistance:")
print(weather_mapping["distance_km"].describe())

(272, 6)
       cell_id  fire_lat  fire_lon  weather_lat  weather_lon  distance_km
0   -4.5_108.0      -4.5    108.00         -4.5       108.00     0.000000
1  -4.5_108.75      -4.5    108.75         -4.5       109.25    55.426074
2   -4.5_109.5      -4.5    109.50         -4.5       109.25    27.713037
3  -4.5_110.25      -4.5    110.25         -4.5       110.50    27.713037
4   -4.5_111.0      -4.5    111.00         -4.5       110.50    55.426074

Distance:
count    272.000000
mean      52.102295
std       22.410521
min        0.000000
25%       39.226991
50%       55.597463
75%       62.154017
max       99.992013
Name: distance_km, dtype: float64


In [35]:
full_fire_panel = full_fire_panel.merge(
    fire_cell_coords,
    on="cell_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", full_fire_panel.shape)
print(full_fire_panel[["cell_id", "lat", "lon"]].head())

Shape: (36448, 20)
       cell_id   lat    lon
0  -0.75_108.0 -0.75  108.0
1  -0.75_108.0 -0.75  108.0
2  -0.75_108.0 -0.75  108.0
3  -0.75_108.0 -0.75  108.0
4  -0.75_108.0 -0.75  108.0


In [36]:
full_fire_panel["date"] = pd.to_datetime(
    full_fire_panel["date"]
)

aq_rich_daily["date"] = pd.to_datetime(
    aq_rich_daily["date"]
)

In [37]:
aq_merge_cols = [
    "lat",
    "lon",
    "date",
    "pm2_5_mean",
    "pm2_5_max",
    "aerosol_optical_depth_mean",
    "carbon_monoxide_mean",
    "formaldehyde_mean"
]

full_fire_panel = full_fire_panel.merge(
    aq_rich_daily[aq_merge_cols],
    on=["lat", "lon", "date"],
    how="left",
    validate="one_to_one"
)

print("Shape setelah AQ:", full_fire_panel.shape)

Shape setelah AQ: (36448, 25)


In [38]:
aq_features = [
    "pm2_5_mean",
    "pm2_5_max",
    "aerosol_optical_depth_mean",
    "carbon_monoxide_mean",
    "formaldehyde_mean"
]

print(full_fire_panel[aq_features].isna().sum())

pm2_5_mean                    0
pm2_5_max                     0
aerosol_optical_depth_mean    0
carbon_monoxide_mean          0
formaldehyde_mean             0
dtype: int64


In [39]:
weather_features = [
    "date",
    "lat",
    "lon",
    "temperature_mean",
    "temperature_max",
    "relative_humidity_mean",
    "precipitation_sum",
    "boundary_layer_height_mean",
    "vapour_pressure_deficit_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_u_mean",
    "wind_v_mean",
    "wind_direction_mean"
]

In [40]:
full_fire_panel = full_fire_panel.merge(
    weather_mapping[
        [
            "cell_id",
            "weather_lat",
            "weather_lon",
            "distance_km"
        ]
    ],
    on="cell_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", full_fire_panel.shape)

Shape: (36448, 28)


In [41]:
weather_merge = weather_rich_daily.copy()

weather_merge["date"] = pd.to_datetime(
    weather_merge["date"]
)

In [42]:
weather_merge_cols = [
    "lat",
    "lon",
    "date",
    "temperature_mean",
    "temperature_max",
    "relative_humidity_mean",
    "precipitation_sum",
    "boundary_layer_height_mean",
    "vapour_pressure_deficit_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_u_mean",
    "wind_v_mean",
    "wind_direction_mean"
]

weather_merge = weather_merge[weather_merge_cols].copy()

weather_merge = weather_merge.rename(
    columns={
        "lat": "weather_lat",
        "lon": "weather_lon"
    }
)

In [44]:
full_fire_panel = full_fire_panel.merge(
    weather_merge,
    on=[
        "weather_lat",
        "weather_lon",
        "date"
    ],
    how="left",
    validate="many_to_one"
)

print("Shape setelah Weather:", full_fire_panel.shape)

Shape setelah Weather: (36448, 39)


In [47]:
weather_feature_cols = [
    "temperature_mean",
    "temperature_max",
    "relative_humidity_mean",
    "precipitation_sum",
    "boundary_layer_height_mean",
    "vapour_pressure_deficit_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_u_mean",
    "wind_v_mean",
    "wind_direction_mean"
]

In [48]:
print("Shape:", full_fire_panel.shape)

print("\nMissing Weather:")
print(
    full_fire_panel[weather_feature_cols].isna().sum()
)

Shape: (36448, 39)

Missing Weather:
temperature_mean                0
temperature_max                 0
relative_humidity_mean          0
precipitation_sum               0
boundary_layer_height_mean      0
vapour_pressure_deficit_mean    0
wind_speed_mean                 0
wind_speed_max                  0
wind_u_mean                     0
wind_v_mean                     0
wind_direction_mean             0
dtype: int64


In [49]:
aq_merge_cols = [
    "lat",
    "lon",
    "date",
    "pm2_5_mean",
    "pm2_5_max",
    "aerosol_optical_depth_mean",
    "carbon_monoxide_mean",
    "formaldehyde_mean"
]

full_fire_panel["date"] = pd.to_datetime(
    full_fire_panel["date"]
)

aq_rich_daily["date"] = pd.to_datetime(
    aq_rich_daily["date"]
)

full_fire_panel = full_fire_panel.merge(
    aq_rich_daily[aq_merge_cols],
    on=["lat", "lon", "date"],
    how="left",
    validate="one_to_one"
)

print("Shape setelah AQ:", full_fire_panel.shape)

Shape setelah AQ: (36448, 44)


In [51]:
print([
    col for col in full_fire_panel.columns
    if any(x in col for x in [
        "pm2_5",
        "aerosol",
        "carbon_monoxide",
        "formaldehyde"
    ])
])

['pm2_5_mean_x', 'pm2_5_max_x', 'aerosol_optical_depth_mean_x', 'carbon_monoxide_mean_x', 'formaldehyde_mean_x', 'pm2_5_mean_y', 'pm2_5_max_y', 'aerosol_optical_depth_mean_y', 'carbon_monoxide_mean_y', 'formaldehyde_mean_y']


In [52]:
aq_pairs = [
    ("pm2_5_mean_x", "pm2_5_mean_y"),
    ("pm2_5_max_x", "pm2_5_max_y"),
    ("aerosol_optical_depth_mean_x", "aerosol_optical_depth_mean_y"),
    ("carbon_monoxide_mean_x", "carbon_monoxide_mean_y"),
    ("formaldehyde_mean_x", "formaldehyde_mean_y")
]

for x, y in aq_pairs:
    print(
        x,
        "sama dengan",
        y,
        ":",
        full_fire_panel[x].equals(full_fire_panel[y])
    )

pm2_5_mean_x sama dengan pm2_5_mean_y : True
pm2_5_max_x sama dengan pm2_5_max_y : True
aerosol_optical_depth_mean_x sama dengan aerosol_optical_depth_mean_y : True
carbon_monoxide_mean_x sama dengan carbon_monoxide_mean_y : True
formaldehyde_mean_x sama dengan formaldehyde_mean_y : True


In [53]:
for x, y in aq_pairs:
    full_fire_panel[x.replace("_x", "")] = full_fire_panel[x]

    full_fire_panel.drop(
        columns=[x, y],
        inplace=True
    )

In [54]:
print([
    col for col in full_fire_panel.columns
    if any(x in col for x in [
        "pm2_5",
        "aerosol",
        "carbon_monoxide",
        "formaldehyde"
    ])
])

print("Shape:", full_fire_panel.shape)

['pm2_5_mean', 'pm2_5_max', 'aerosol_optical_depth_mean', 'carbon_monoxide_mean', 'formaldehyde_mean']
Shape: (36448, 39)


In [55]:
print(full_fire_panel.columns.tolist())

['cell_id', 'date', 'n_detections', 'frp_sum', 'frp_mean', 'frp_max', 'fire_active', 'target_fire_active_t1', 'fire_count_lag1', 'fire_count_lag3', 'fire_count_lag7', 'frp_sum_lag1', 'frp_sum_lag3', 'frp_sum_lag7', 'fire_count_roll3', 'fire_count_roll7', 'frp_sum_roll3', 'frp_sum_roll7', 'lat', 'lon', 'weather_lat', 'weather_lon', 'distance_km', 'temperature_mean', 'temperature_max', 'relative_humidity_mean', 'precipitation_sum', 'boundary_layer_height_mean', 'vapour_pressure_deficit_mean', 'wind_speed_mean', 'wind_speed_max', 'wind_u_mean', 'wind_v_mean', 'wind_direction_mean', 'pm2_5_mean', 'pm2_5_max', 'aerosol_optical_depth_mean', 'carbon_monoxide_mean', 'formaldehyde_mean']


In [56]:
feature_cols = [
    # Fire history
    "fire_count_lag1",
    "fire_count_lag3",
    "fire_count_lag7",
    "frp_sum_lag1",
    "frp_sum_lag3",
    "frp_sum_lag7",
    "fire_count_roll3",
    "fire_count_roll7",
    "frp_sum_roll3",
    "frp_sum_roll7",

    # Weather
    "temperature_mean",
    "temperature_max",
    "relative_humidity_mean",
    "precipitation_sum",
    "boundary_layer_height_mean",
    "vapour_pressure_deficit_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_u_mean",
    "wind_v_mean",

    # Air Quality
    "pm2_5_mean",
    "pm2_5_max",
    "aerosol_optical_depth_mean",
    "carbon_monoxide_mean",
    "formaldehyde_mean"
]

target_col = "target_fire_active_t1"

In [57]:
print("Jumlah fitur:", len(feature_cols))

print("\nMissing features:")
print(full_fire_panel[feature_cols].isna().sum())

print("\nMissing target:")
print(full_fire_panel[target_col].isna().sum())

Jumlah fitur: 25

Missing features:
fire_count_lag1                  272
fire_count_lag3                  816
fire_count_lag7                 1904
frp_sum_lag1                     272
frp_sum_lag3                     816
frp_sum_lag7                    1904
fire_count_roll3                 816
fire_count_roll7                1904
frp_sum_roll3                    816
frp_sum_roll7                   1904
temperature_mean                   0
temperature_max                    0
relative_humidity_mean             0
precipitation_sum                  0
boundary_layer_height_mean         0
vapour_pressure_deficit_mean       0
wind_speed_mean                    0
wind_speed_max                     0
wind_u_mean                        0
wind_v_mean                        0
pm2_5_mean                         0
pm2_5_max                          0
aerosol_optical_depth_mean         0
carbon_monoxide_mean               0
formaldehyde_mean                  0
dtype: int64

Missing target:
272


In [58]:
ml_data = full_fire_panel.dropna(
    subset=feature_cols + [target_col]
).copy()

print("Shape:", ml_data.shape)

print(
    "Tanggal:",
    ml_data["date"].min(),
    "→",
    ml_data["date"].max()
)

print(
    "Jumlah cell:",
    ml_data["cell_id"].nunique()
)

print("\nTarget:")
print(ml_data[target_col].value_counts())

Shape: (34272, 39)
Tanggal: 2026-05-08 00:00:00 → 2026-09-10 00:00:00
Jumlah cell: 272

Target:
target_fire_active_t1
0.0    27990
1.0     6282
Name: count, dtype: int64


In [59]:
print(
    "Total missing features:",
    ml_data[feature_cols].isna().sum().sum()
)

print(
    "Missing target:",
    ml_data[target_col].isna().sum()
)

Total missing features: 0
Missing target: 0


In [60]:
print("Feature count:", len(feature_cols))

print("\nFeatures:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i:2}. {col}")

Feature count: 25

Features:
 1. fire_count_lag1
 2. fire_count_lag3
 3. fire_count_lag7
 4. frp_sum_lag1
 5. frp_sum_lag3
 6. frp_sum_lag7
 7. fire_count_roll3
 8. fire_count_roll7
 9. frp_sum_roll3
10. frp_sum_roll7
11. temperature_mean
12. temperature_max
13. relative_humidity_mean
14. precipitation_sum
15. boundary_layer_height_mean
16. vapour_pressure_deficit_mean
17. wind_speed_mean
18. wind_speed_max
19. wind_u_mean
20. wind_v_mean
21. pm2_5_mean
22. pm2_5_max
23. aerosol_optical_depth_mean
24. carbon_monoxide_mean
25. formaldehyde_mean


In [61]:
ml_data.to_csv(
    "../data/processed/karhut_ml_dataset.csv",
    index=False
)

print("Final ML dataset saved!")

Final ML dataset saved!


In [62]:
print("Total missing:", ml_data[feature_cols].isna().sum().sum())
print("Missing target:", ml_data[target_col].isna().sum())

Total missing: 0
Missing target: 0
